Identification of MicroAge-based age-decelerated and age-accelerated host features

To define the MicroAge-based age-decelerated and age-accelerated host features, MaAsLin2 (version 1.16.0) was employed. The aging pace (MicroAge pace) was defined as the residual between the predicted age and its linear regression value with chronological age. Enterotype, sex, BMI, and age were included as covariates for correction, and age pace was fitted into the model.
Feature ≈ MicroAge pace + age + enterotype + sex + BMI
Consistent with prior studies, adjusted P value less than 0.1 was set as the threshold for statistical significance28,93. Specifically, host features with a positive coefficient were defined as age‑decelerated features, whereas those with a negative coefficient were defined as age‑accelerated features.

In [ ]:
rm(list=ls())
library(ggsci)
library(dplyr)
library(patchwork)
library(Maaslin2)
library(vegan)
library(edgeR)
library(tidyverse)
library(ggplot2)
library(factoextra)
library(ggpubr)
library(FactoMineR)
library(edgeR)
library(pls)
library(Seurat)

rawdata <- read.csv("FinalPredAge.csv")
names(rawdata)
table(rawdata$Group)

aging_train <- rawdata%>%filter(Group=="Aging")%>%filter(source =="train")
aging <- rawdata%>%filter(Group=="Aging")%>%filter(source =="valid")

table(aging_train$Group)

CA=aging_train$Age
predAge=aging_train$PredAge
lm = lm(predAge~CA)

final<-rawdata%>%
  mutate(delta_Age = PredAge - (Age * lm$coefficients[2] + lm$coefficients[1]))%>%filter(source =="valid")%>%
  dplyr::select(delta_Age,id)

df_input_metadata = read.table(file             = "E:/QZ/QZ_metagenomics/04_Merge/Sample_metadata.csv", 
                               header           = TRUE, 
                               sep              = ",", 
                               row.names        = 14,
                               stringsAsFactors = FALSE)

df_input_metadata<-df_input_metadata%>%dplyr::filter(Group %in% c("Control"))
table(df_input_metadata$Group)
table(df_input_metadata$Sex)

df<-merge(final,df_input_metadata,by.x="id",by.y="Name_metaphlan")
df[is.na(df)]<- 0

df2<-df%>%mutate(status=ifelse(delta_Age<quantile(df$delta_Age,probs = seq(0,1,0.1))[4],"Decelerated",
                               ifelse(delta_Age>quantile(df$delta_Age,probs = seq(0,1,0.1))[8],"accelerated","None")))
    
write.csv(df2,"Clock_sample_selected.csv")